In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.io as io
from torchvision import transforms
from mamba_ssm import Mamba

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


/home/imvikranth/mamba_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


In [2]:
ROOT = "/home/imvikranth/mini_project/datasets/UCF101"

TRAIN_DIR = os.path.join(ROOT, "train")
VAL_DIR   = os.path.join(ROOT, "val")

TRAIN_CSV = os.path.join(ROOT, "train.csv")
VAL_CSV   = os.path.join(ROOT, "val.csv")


In [3]:
df = pd.read_csv(TRAIN_CSV)
print(df.head())
print(df.columns)


         clip_name                         clip_path  label
0  v_Swing_g05_c02  /train/Swing/v_Swing_g05_c02.avi  Swing
1  v_Swing_g21_c03  /train/Swing/v_Swing_g21_c03.avi  Swing
2  v_Swing_g07_c01  /train/Swing/v_Swing_g07_c01.avi  Swing
3  v_Swing_g24_c04  /train/Swing/v_Swing_g24_c04.avi  Swing
4  v_Swing_g20_c03  /train/Swing/v_Swing_g20_c03.avi  Swing
Index(['clip_name', 'clip_path', 'label'], dtype='str')


In [4]:
df = pd.read_csv(TRAIN_CSV)
print(df.head())
print(df.columns)

         clip_name                         clip_path  label
0  v_Swing_g05_c02  /train/Swing/v_Swing_g05_c02.avi  Swing
1  v_Swing_g21_c03  /train/Swing/v_Swing_g21_c03.avi  Swing
2  v_Swing_g07_c01  /train/Swing/v_Swing_g07_c01.avi  Swing
3  v_Swing_g24_c04  /train/Swing/v_Swing_g24_c04.avi  Swing
4  v_Swing_g20_c03  /train/Swing/v_Swing_g20_c03.avi  Swing
Index(['clip_name', 'clip_path', 'label'], dtype='str')


In [5]:
class UCF101FrameDataset(Dataset):
    def __init__(self, csv_file, root_dir, frames_per_video=10, transform=None):
        self.df = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.frames_per_video = frames_per_video
        self.transform = transform

        self.video_paths = self.df["clip_path"].values
        self.labels_str = self.df["label"].values

        self.classes = sorted(self.df["label"].unique())
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

    def __len__(self):
        return len(self.video_paths) * self.frames_per_video

    def __getitem__(self, index):
        video_idx = index // self.frames_per_video
        frame_idx = index % self.frames_per_video

        rel_path = self.video_paths[video_idx].lstrip("/")
        full_path = os.path.join(self.root_dir, rel_path)

        label_str = self.labels_str[video_idx]
        label = self.class_to_idx[label_str]

        video, _, _ = io.read_video(full_path, pts_unit="sec")
        total_frames = video.shape[0]

        indices = torch.linspace(0, total_frames - 1, self.frames_per_video).long()
        frame = video[indices[frame_idx]]

        frame = frame.permute(2, 0, 1).float() / 255.0

        if self.transform:
            frame = self.transform(frame)

        return frame, label

In [10]:
train_dataset = UCF101FrameDataset(TRAIN_CSV, ROOT, frames_per_video=10, transform=transform)

In [11]:
frame, label = train_dataset[0]
print(frame.shape, label)

/home/imvikranth/mamba_env/lib/python3.11/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


torch.Size([3, 128, 128]) 88


In [16]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
])

train_dataset = UCF101FrameDataset(
    TRAIN_CSV,
    ROOT,
    frames_per_video=10,
    transform=transform
)

val_dataset = UCF101FrameDataset(
    VAL_CSV,
    ROOT,
    frames_per_video=10,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

num_classes = len(train_dataset.classes)

print("Classes:", num_classes)

Classes: 101


In [17]:
class MambaImageClassifier(nn.Module):
    def __init__(self, num_classes, d_model=128, patch_size=16):
        super().__init__()

        self.patch_embed = nn.Conv2d(3, d_model, kernel_size=patch_size, stride=patch_size)

        self.mamba = Mamba(
            d_model=d_model,
            d_state=16,
            d_conv=4,
            expand=2,
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.mamba(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.head(x)


In [18]:
model = MambaImageClassifier(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)


In [19]:
for epoch in range(3):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")


/home/imvikranth/mamba_env/lib/python3.11/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
/home/imvikranth/mamba_env/lib/python3.11/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(
/home/imvikranth/mamba_env/lib/python3.11/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding c

KeyboardInterrupt: 

### Method -2 : Pre-extract 7 frames from each video before training the model can reduce the training from hours to mins

In [21]:
import os
import pandas as pd
import torch
import torchvision.io as io
from torchvision.utils import save_image

ROOT = "/home/imvikranth/mini_project/datasets/UCF101"
OUTPUT_ROOT = "/home/imvikranth/mini_project/frame_dataset"

FRAMES_PER_VIDEO = 7

os.makedirs(OUTPUT_ROOT, exist_ok=True)

def extract_frames(csv_file, split_name):
    df = pd.read_csv(csv_file)

    for idx, row in df.iterrows():
        rel_path = row["clip_path"].lstrip("/")
        label = row["label"]

        video_path = os.path.join(ROOT, rel_path)

        try:
            video, _, _ = io.read_video(video_path, pts_unit="sec")
        except:
            print("Failed:", video_path)
            continue

        total_frames = video.shape[0]
        if total_frames < FRAMES_PER_VIDEO:
            continue

        indices = torch.linspace(0, total_frames - 1, FRAMES_PER_VIDEO).long()

        save_dir = os.path.join(OUTPUT_ROOT, split_name, label)
        os.makedirs(save_dir, exist_ok=True)

        for i, frame_idx in enumerate(indices):
            frame = video[frame_idx].permute(2, 0, 1).float() / 255.0
            save_path = os.path.join(
                save_dir,
                f"{os.path.basename(video_path)}_f{i}.jpg"
            )
            save_image(frame, save_path)

        if idx % 200 == 0:
            print(f"{split_name}: Processed {idx}/{len(df)}")

# Extract
extract_frames(os.path.join(ROOT, "train.csv"), "train")
extract_frames(os.path.join(ROOT, "val.csv"), "val")

/home/imvikranth/mamba_env/lib/python3.11/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


train: Processed 0/10055
train: Processed 200/10055
train: Processed 400/10055
train: Processed 600/10055
train: Processed 800/10055
train: Processed 1000/10055
train: Processed 1200/10055
train: Processed 1400/10055
train: Processed 1600/10055
train: Processed 1800/10055
train: Processed 2000/10055
train: Processed 2200/10055
train: Processed 2400/10055
train: Processed 2600/10055
train: Processed 2800/10055
train: Processed 3000/10055
train: Processed 3200/10055
train: Processed 3400/10055
train: Processed 3600/10055
train: Processed 3800/10055
train: Processed 4000/10055
train: Processed 4200/10055
train: Processed 4400/10055
train: Processed 4600/10055
train: Processed 4800/10055
train: Processed 5000/10055
train: Processed 5200/10055
train: Processed 5400/10055
train: Processed 5600/10055
train: Processed 5800/10055
train: Processed 6000/10055
train: Processed 6200/10055
train: Processed 6400/10055
train: Processed 6600/10055
train: Processed 6800/10055
train: Processed 7000/10055

/home/imvikranth/mamba_env/lib/python3.11/site-packages/torchvision/io/_video_deprecation_warning.py:5: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


val: Processed 0/1673
val: Processed 200/1673
val: Processed 400/1673
val: Processed 600/1673
val: Processed 800/1673
val: Processed 1000/1673
val: Processed 1200/1673
val: Processed 1400/1673
val: Processed 1600/1673


In [31]:
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

train_dataset = ImageFolder(
    root="/home/imvikranth/mini_project/frame_dataset/train",
    transform=transform
)

val_dataset = ImageFolder(
    root="/home/imvikranth/mini_project/frame_dataset/val",
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

num_classes = len(train_dataset.classes)
print("Classes:", num_classes)

Classes: 101


In [32]:
import torch.nn as nn
from mamba_ssm import Mamba

class MambaImageClassifier(nn.Module):
    def __init__(self, num_classes, d_model=128, patch_size=16):
        super().__init__()

        self.patch_embed = nn.Conv2d(3, d_model, kernel_size=patch_size, stride=patch_size)

        self.mamba = Mamba(
            d_model=d_model,
            d_state=16,
            d_conv=4,
            expand=2,
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.mamba(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        return self.head(x)

In [33]:
import torch
import torch.optim as optim

device = "cuda" if torch.cuda.is_available() else "cpu"

model = MambaImageClassifier(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")

Epoch 1, Loss: 3.7688
Epoch 2, Loss: 2.7754
Epoch 3, Loss: 2.1740
Epoch 4, Loss: 1.7270
Epoch 5, Loss: 1.4003
Epoch 6, Loss: 1.1733
Epoch 7, Loss: 1.0046
Epoch 8, Loss: 0.8833
Epoch 9, Loss: 0.7864
Epoch 10, Loss: 0.7088
Epoch 11, Loss: 0.6528
Epoch 12, Loss: 0.5927
Epoch 13, Loss: 0.5550
Epoch 14, Loss: 0.5155
Epoch 15, Loss: 0.4875
Epoch 16, Loss: 0.4563
Epoch 17, Loss: 0.4345
Epoch 18, Loss: 0.4108
Epoch 19, Loss: 0.3909
Epoch 20, Loss: 0.3748


Loss in each epoch as recorded : 
Epoch 1, Loss: 3.7688
Epoch 2, Loss: 2.7754
Epoch 3, Loss: 2.1740
Epoch 4, Loss: 1.7270
Epoch 5, Loss: 1.4003
Epoch 6, Loss: 1.1733
Epoch 7, Loss: 1.0046
Epoch 8, Loss: 0.8833
Epoch 9, Loss: 0.7864
Epoch 10, Loss: 0.7088
Epoch 11, Loss: 0.6528
Epoch 12, Loss: 0.5927
Epoch 13, Loss: 0.5550
Epoch 14, Loss: 0.5155
Epoch 15, Loss: 0.4875
Epoch 16, Loss: 0.4563
Epoch 17, Loss: 0.4345
Epoch 18, Loss: 0.4108
Epoch 19, Loss: 0.3909
Epoch 20, Loss: 0.3748

Total time taken : 11m 39.8 s

In [1]:
model.eval()

images, labels = next(iter(val_loader))
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    outputs = model(images)
    preds = outputs.argmax(dim=1)

print("Predictions:", preds[:].cpu().numpy())
print("Ground Truth:", labels[:].cpu().numpy())

NameError: name 'model' is not defined

Predictions: [0 0 0 0 0 0 0 0 0 0]
Ground Truth: [0 0 0 0 0 0 0 0 0 0]

In [35]:
import torch
from PIL import Image
from torchvision import transforms

img_path = "/home/imvikranth/mini_project/frame_dataset/train/Swing/v_Swing_g01_c03.avi_f3.jpg"

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

image = Image.open(img_path).convert("RGB")
image = transform(image).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    output = model(image)
    pred = output.argmax(dim=1).item()

print("Predicted class:", train_dataset.classes[pred])

Predicted class: Swing


Predicted class: Swing

In [36]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Validation Accuracy:", correct / total)

Validation Accuracy: 0.7643241396977201


Validation Accuracy: 0.7643241396977201